In [9]:
import pandas as pd
import numpy as np
import re
import os
from arabert.preprocess import ArabertPreprocessor

RAW_DIR = '../data/raw/'
PROCESSED_DIR = '../data/processed/'
os.makedirs(PROCESSED_DIR, exist_ok=True)

df_train = pd.read_parquet(RAW_DIR + 'train-00000-of-00001.parquet')
df_valid = pd.read_parquet(RAW_DIR + 'validation-00000-of-00001.parquet')
df_test = pd.read_parquet(RAW_DIR + 'test-00000-of-00001.parquet')

print(f"Original Size - Train: {df_train.shape}, Valid: {df_valid.shape}, Test: {df_test.shape}")

Original Size - Train: (54845, 15), Valid: (7310, 15), Test: (7286, 15)


In [10]:
train_initial_len = len(df_train)
df_train = df_train[df_train['Sentence'] != '#NAME?'].copy()
print(f" Deleted {train_initial_len - len(df_train)} error columns '#NAME?' in the training set.")

for df in [df_train, df_valid, df_test]:
    df.dropna(subset=['Sentence'], inplace=True)

 Deleted 219 error columns '#NAME?' in the training set.


In [11]:
# Transform Readability_Level_19 to label (0-18)
for df in [df_train, df_valid, df_test]:
    if 'Readability_Level_19' in df.columns:
        df['label'] = df['Readability_Level_19'].astype(int) - 1

print(" Created 'label' column (0-18)!")

 Created 'label' column (0-18)!


In [12]:
# Function to calculate the ratio of diacritics in a given Arabic text
arabic_diacritics = re.compile("""
                             ّ    | َ    | ً    | ُ    | ٌ    | ِ    | ٍ    | ْ
                         """, re.VERBOSE)

def calculate_diacritics_ratio(text):
    if not isinstance(text, str) or len(text) == 0: return 0.0
    return len(re.findall(arabic_diacritics, text)) / len(text)

# Function to initialize the main preprocessing pipeline for AraBERT
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
arabert_prep = ArabertPreprocessor(model_name=MODEL_NAME)

def normalize_for_arabert(text):
    if not isinstance(text, str): return ""
    return arabert_prep.preprocess(text)

In [13]:
for df_name, df in zip(['Train', 'Valid', 'Test'], [df_train, df_valid, df_test]):
    
    df['Diacritics_Ratio'] = df['Sentence'].apply(calculate_diacritics_ratio)
    
    df['Sentence_Normalized'] = df['Sentence'].apply(normalize_for_arabert)
    print(f"Finished processing {df_name} set")

display(df_train[['Sentence', 'Sentence_Normalized', 'Diacritics_Ratio', 'label']].head())

Finished processing Train set
Finished processing Valid set
Finished processing Test set


,Sentence,Sentence_Normalized,Diacritics_Ratio,label
0,مجلة كل الأولاد وكل البنات,مجلة كل الأولاد وكل البنات,0.0,6
1,ماجد,ماجد,0.0,0
2,الأربعاء 21 يناير 1987,الأربعاء 21 يناير 1987,0.0,7
3,الموافق 21 جمادى الأول 1407هــ,الموافق 21 جمادى الأول 1407 ه,0.0,6
4,السنة الثامنة,السنة الثامنة,0.0,4


In [14]:
df_train.to_parquet(PROCESSED_DIR + 'train_cleaned.parquet', index=False)
df_valid.to_parquet(PROCESSED_DIR + 'valid_cleaned.parquet', index=False)
df_test.to_parquet(PROCESSED_DIR + 'test_cleaned.parquet', index=False)

print(f" Done! Data has been preprocessed specifically for AraBERT and saved at: {PROCESSED_DIR}")

 Done! Data has been preprocessed specifically for AraBERT and saved at: ../data/processed/


In [15]:
print("Checking for duplicates and label conflicts across the entire dataset...")

df_train['split_tag'] = 'train'
df_valid['split_tag'] = 'valid'
df_test['split_tag'] = 'test'

df_all = pd.concat([df_train, df_valid, df_test], ignore_index=True)
initial_len = len(df_all)

# 1. Delete Hard Contradictions
dups = df_all[df_all.duplicated(subset=['Sentence_Normalized'], keep=False)]

# Count the number of unique labels for each sentence
conflict_sents = dups.groupby('Sentence_Normalized')['label'].nunique()
conflict_list = conflict_sents[conflict_sents > 1].index

# Loại bỏ hoàn toàn các câu có xung đột (cùng 1 câu nhưng giám khảo gán 2 nhãn khác nhau)
df_all = df_all[~df_all['Sentence_Normalized'].isin(conflict_list)].copy()
conflict_dropped = initial_len - len(df_all)

# 2. Delete Exact Duplicates
# Same content, same label -> Keep only the first occurrence (prioritize keeping in Train set since it was concatenated first)
df_all = df_all.drop_duplicates(subset=['Sentence_Normalized'], keep='first').copy()
exact_dropped = initial_len - conflict_dropped - len(df_all)

print(f" Deleted {conflict_dropped} rows with label conflicts.")
print(f" Deleted {exact_dropped} rows with exact duplicates (Reducing data leakage and improving training speed).")
print(f" Total clean data remaining: {len(df_all)} samples.\n")

# 3. SPLIT BACK INTO THE ORIGINAL 3 SETS
df_train = df_all[df_all['split_tag'] == 'train'].drop(columns=['split_tag']).copy()
df_valid = df_all[df_all['split_tag'] == 'valid'].drop(columns=['split_tag']).copy()
df_test  = df_all[df_all['split_tag'] == 'test'].drop(columns=['split_tag']).copy()

print(f" Size after deep cleaning - Train: {df_train.shape}, Valid: {df_valid.shape}, Test: {df_test.shape}")

Checking for duplicates and label conflicts across the entire dataset...
 Deleted 10 rows with label conflicts.
 Deleted 3972 rows with exact duplicates (Reducing data leakage and improving training speed).
 Total clean data remaining: 65240 samples.

 Size after deep cleaning - Train: (51618, 18), Valid: (6877, 18), Test: (6745, 18)
